# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ma5029blp-wq/ML-flyrank-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import pandas as pd
import numpy as np

DATA_PATH = "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

## 1. Distributions

I first checked the distributions of the three signals I plan to audit: days since last update, impression visibility, and position. Traffic and performance metrics are heavy-tailed, so I use bucketed summaries rather than relying only on raw averages.

The data show that days since last update is concentrated around lower values but has a long tail, while impression and position measures also contain substantial variation. These distributions motivate using readable buckets for the signal tests.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Distributions of the signals we will audit

print("Days since last update:")
print(df["days_since_last_update"].describe())

print("\nImpressions - last 30 days:")
print(df["impressions_last_30d"].describe())

print("\nImpressions - previous 30 days:")
print(df["impressions_prev_30d"].describe())

print("\nAverage position:")
print(df["avg_position"].describe())

print("\nPosition tiers:")
print(df["position_tier"].value_counts(dropna=False))

print("\nImpression tiers:")
print(df["impression_tier"].value_counts(dropna=False))

Days since last update:
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%        104.000000
max        373.000000
Name: days_since_last_update, dtype: float64

Impressions - last 30 days:
count     30000.000000
mean       1429.058733
std        5643.852081
min           0.000000
25%          10.000000
50%         139.000000
75%         768.000000
max      238796.000000
Name: impressions_last_30d, dtype: float64

Impressions - previous 30 days:
count     30000.000000
mean       1783.078500
std        6150.429511
min           0.000000
25%          19.000000
50%         210.000000
75%        1143.000000
max      218786.000000
Name: impressions_prev_30d, dtype: float64

Average position:
count    30000.00000
mean        16.34238
std         15.21679
min          0.00000
25%          6.20000
50%         10.80000
75%         22.30000
max        245.00000
Name: avg_position, dtype: float64

Position tiers:


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal Test #1 — Staleness

**Claim:** Older content is more likely to be declining.

**Test:** Compare the observed decline rate across buckets of days since the last update, while showing the number of pages (`n`) in each bucket.

**Verdict:** MIXED — the observed decline rate increases from the younger buckets to the 91–180 day bucket, but falls substantially in the 181+ bucket. This suggests staleness may be useful directionally, but the relationship is not consistent across all buckets.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Create the observed decline indicator
df["declining_observed"] = (
    df["impressions_last_30d"] < 0.8 * df["impressions_prev_30d"]
).astype(int)

print(df["declining_observed"].value_counts())

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0, 30, 90, 180, float("inf")],
    labels=["0-30 days", "31-90 days", "91-180 days", "181+ days"]
)

staleness_test = (
    df.groupby("staleness_bucket", observed=True)
      .agg(
          n=("declining_observed", "size"),
          declining_rate=("declining_observed", "mean")
      )
      .reset_index()
)

staleness_test["declining_rate_pct"] = (
    staleness_test["declining_rate"] * 100
).round(1)

print(staleness_test)

declining_observed
1    16262
0    13738
Name: count, dtype: int64
  staleness_bucket      n  declining_rate  declining_rate_pct
0        0-30 days  20480        0.511377                51.1
1       31-90 days    175        0.588571                58.9
2      91-180 days   9171        0.611057                61.1
3        181+ days    174        0.471264                47.1


### Signal Test #2 — Impression Visibility

**Claim:** Pages with greater impression visibility are more likely to be declining.

**Test:** Compare the observed decline rate across impression tiers and report the number of pages (`n`) in each tier.

**Verdict:** MIXED — moderate and good impression tiers have higher observed decline rates than the low tier, but the excellent tier falls back close to the low tier. This means impression visibility shows a directional difference in some buckets, but the relationship is not consistently increasing.

In [13]:
visibility_test = (
    df.groupby("impression_tier", observed=True)
      .agg(
          n=("declining_observed", "size"),
          declining_rate=("declining_observed", "mean")
      )
      .reset_index()
)

visibility_test["declining_rate_pct"] = (
    visibility_test["declining_rate"] * 100
).round(1)

print(visibility_test)

  impression_tier      n  declining_rate  declining_rate_pct
0       excellent   1078        0.461967                46.2
1            good   7205        0.586121                58.6
2             low  11248        0.453947                45.4
3        moderate  10469        0.614672                61.5


### Signal Test #3 — Search Position

**Claim:** Search position is related to whether a page is declining.

**Test:** Compare the observed decline rate across position tiers and report the number of pages (`n`) in each tier. The result is interpreted cautiously because position can be noisy when impression volume is low.

**Verdict:** MIXED — decline rates differ across position tiers, but they do not follow a simple pattern. The striking tier has the highest observed decline rate, while the top-3 tier has a much lower decline rate. Position therefore appears directionally useful, but the tiers alone do not show a consistent relationship with decline.

In [14]:
position_test = (
    df.groupby("position_tier", observed=True)
      .agg(
          n=("declining_observed", "size"),
          declining_rate=("declining_observed", "mean")
      )
      .reset_index()
)

position_test["declining_rate_pct"] = (
    position_test["declining_rate"] * 100
).round(1)

print(position_test)

  position_tier      n  declining_rate  declining_rate_pct
0          deep   1319        0.344200                34.4
1        page_1  11814        0.569663                57.0
2      page_3_5   7242        0.561585                56.2
3      striking   7304        0.609529                61.0
4         top_3   2321        0.240844                24.1


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

## 3. The Flag-Linked Test

**Flag-linked signal:** Staleness.

**Claim:** The refresh flag's use of staleness assumes that stale content is more likely to show a decline.

**Test:** Compare pages updated less than 91 days ago with pages that have been stale for 91+ days. Report the number of pages (`n`) and the observed decline rate for each group.

**Verdict:** CONFIRMED — pages that were stale for 91+ days had a higher observed decline rate (60.8%) than pages updated within the previous 90 days (51.2%). This supports the direction of the refresh flag, although staleness should be treated as supporting evidence rather than proof that a page needs a refresh.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Flag-linked test: staleness

df["stale_91_plus"] = (
    df["days_since_last_update"] >= 91
)

flag_staleness_test = (
    df.groupby("stale_91_plus")
      .agg(
          n=("declining_observed", "size"),
          declining_rate=("declining_observed", "mean")
      )
      .reset_index()
)

flag_staleness_test["declining_rate_pct"] = (
    flag_staleness_test["declining_rate"] * 100
).round(1)

print(flag_staleness_test)

   stale_91_plus      n  declining_rate  declining_rate_pct
0          False  20655        0.512031                51.2
1           True   9345        0.608454                60.8


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

###A:-
The audit suggests that staleness is useful as a supporting signal for refresh decisions, especially at the 91+ day threshold, where the observed decline rate was higher. Impression visibility and search position showed mixed patterns, so a content team should use them as decision-support signals rather than treating them as proof that a page needs action.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.